# ERA5 — cubo di anomalie ed eventi spazio-temporali

Analisi separata dal training STGAN. Default: **demo sintetica**, nessun download o training.
Per risultati reali, eseguire prima `scripts/run_era5_stgan.py events`, poi impostare `USE_SYNTHETIC_DEMO=False` e `EVENT_DIR`.
Le funzioni di calcolo e plotting sono nei moduli `physiq_pv.era5`; le matrici vengono aperte con memmap e visualizzate un timestamp alla volta.

Configurazione ERA5: 15 feature, griglia 0.5°, passo 3 h; training 1980–2004, scoring 2005–ultimo anno completo locale.
Le celle escluse per dati mancanti rimangono NaN. La soglia percentile globale usa tutti gli score finiti e confronto stretto `>`.
Gli eventi sono componenti del grafo temporale: split e merge mantengono gli archi tra cluster e possono unificare retroattivamente gli event ID.

In [ ]:
from pathlib import Path
import sys, json, sqlite3
from contextlib import closing
import pandas as pd
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'physiq_pv').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from physiq_pv.era5.visualization import create_synthetic_demo, plot_frame, plot_event, plot_event_snapshots
USE_SYNTHETIC_DEMO = True
EVENT_DIR = ROOT / 'outputs/era5/events_top1'
if USE_SYNTHETIC_DEMO:
    EVENT_DIR = create_synthetic_demo(ROOT / 'outputs/era5/synthetic_cube_demo')
metadata = json.loads((EVENT_DIR / 'metadata.json').read_text(encoding='utf-8'))
assert metadata['status'] == 'complete', 'Elaborazione eventi incompleta'
display(pd.Series(metadata['config'], name='Configurazione eventi'))
print('Shape (T,H,W):', metadata['shape'])
print('Cluster:', metadata['n_clusters'], '| Eventi:', metadata['n_events'])

In [ ]:
with closing(sqlite3.connect(EVENT_DIR / 'events.sqlite')) as db:
    events = pd.read_sql_query('SELECT * FROM events ORDER BY max_score DESC LIMIT 30', db)
display(events)

## Un timestamp
Modificare `TIME_INDEX` per confrontare score, threshold, opening, closing e cluster. Gli ID dei cluster sono globali, mentre gli event ID sono consultabili nella tabella. I bordi esterni e le celle senza dati restano sfondo nella morfologia.

In [ ]:
TIME_INDEX = min(4, metadata['shape'][0] - 1)
figure, clusters = plot_frame(EVENT_DIR, TIME_INDEX)
display(clusters)

## Evoluzione di un evento
La traiettoria mostra il centroide pesato per area e area/score nel tempo. Se un evento si divide, `event_steps` aggrega i rami nello stesso timestamp; `clusters` e `links` conservano i singoli rami e le relazioni split/merge. La durata include l'intervallo dell'ultimo campione; `elapsed_hours` riporta invece `end - start`.

In [ ]:
EVENT_ID = int(events.event_id.iloc[0]) if len(events) else None
if EVENT_ID is not None:
    figure, trajectory = plot_event(EVENT_DIR, EVENT_ID)
    display(trajectory)
    snapshots = plot_event_snapshots(EVENT_DIR, EVENT_ID)
    with closing(sqlite3.connect(EVENT_DIR / 'events.sqlite')) as db:
        links = pd.read_sql_query(
            'SELECT l.* FROM links l JOIN clusters c ON c.cluster_id=l.source_cluster WHERE c.event_id=? LIMIT 200',
            db, params=(EVENT_ID,))
    display(links)
else:
    print('Nessun evento con questa configurazione.')